# CatCare AI — treinamento no Colab

Este notebook treina o EfficientNet-B0 e baixa o checkpoint para usar no backend. Execute as células em ordem. Antes de começar, escolha **Ambiente de execução → Alterar tipo de ambiente de execução → GPU**, se houver GPU disponível.

## O que enviar

Prepare **um arquivo ZIP** com imagens e um `labels.csv` que descreva cada imagem. O dataset é separado das fotos enviadas por usuários ao aplicativo. **Não envie `.env` nem chaves de API.**

```text
dataset.zip
└── dataset/
    ├── labels.csv
    └── images/
        ├── gato_001.jpg
        └── gato_002.jpg
```

O CSV precisa das colunas `path,breed,features,coat_pattern,colors,coat_length,split`. Exemplo ilustrativo (o treino real precisa de muitas imagens rotuladas, incluindo exemplos de treino e validação):

```csv
path,breed,features,coat_pattern,colors,coat_length,split
images/gato_001.jpg,srd,frajola|bicolor,tuxedo,black|white,short,train
images/gato_002.jpg,siamese,sialata,colorpoint,cream|brown,short,val
```

As listas de características e cores usam `|`. Cada linha precisa de rótulos válidos para todas as saídas; fotos soltas não são suficientes para treinar o modelo atual.


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO = Path('/content/catcare-ai')
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Danilogggs/gatitos.git', str(REPO)], check=True)
sys.path.insert(0, str(REPO / 'backend'))
from app.ml.taxonomy import BREEDS, FEATURES, PATTERNS, COLORS, COAT_LENGTHS
print('Raças:', ', '.join(BREEDS))
print('Características:', ', '.join(FEATURES))
print('Padrões:', ', '.join(PATTERNS))
print('Cores:', ', '.join(COLORS))
print('Comprimentos:', ', '.join(COAT_LENGTHS))


## Envie o ZIP do dataset

Ao executar a próxima célula, o Colab abrirá o seletor de arquivos. Escolha somente o ZIP do dataset rotulado.


In [ ]:
from google.colab import files
from zipfile import ZipFile

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Envie exatamente um arquivo ZIP.')
archive = Path('/content') / next(iter(uploaded))
if archive.suffix.lower() != '.zip':
    raise ValueError('O arquivo precisa ser ZIP.')
DATA_DIR = Path('/content/catcare_dataset')
DATA_DIR.mkdir(exist_ok=True)
with ZipFile(archive) as zipped:
    for entry in zipped.infolist():
        member = Path(entry.filename)
        if member.is_absolute() or '..' in member.parts:
            raise ValueError('O ZIP contém um caminho inválido.')
    zipped.extractall(DATA_DIR)
csv_files = list(DATA_DIR.rglob('labels.csv'))
if len(csv_files) != 1:
    raise ValueError('O ZIP deve conter exatamente um labels.csv.')
CSV_PATH = csv_files[0]
print('Dataset recebido:', CSV_PATH)


## Confira os rótulos antes do treino

Esta célula verifica o formato do CSV, as classes e os caminhos das imagens. Corrija o dataset se aparecer algum erro.


In [ ]:
import csv
from collections import Counter

columns = {'path', 'breed', 'features', 'coat_pattern', 'colors', 'coat_length', 'split'}
with CSV_PATH.open(newline='', encoding='utf-8-sig') as handle:
    reader = csv.DictReader(handle)
    if not columns.issubset(reader.fieldnames or []):
        raise ValueError(f'Colunas ausentes: {sorted(columns - set(reader.fieldnames or []))}')
    rows = list(reader)
if not rows:
    raise ValueError('O CSV está vazio.')
root = CSV_PATH.parent.resolve()
for line, row in enumerate(rows, start=2):
    path = (root / row['path']).resolve()
    if not path.is_relative_to(root) or not path.is_file():
        raise ValueError(f'Linha {line}: imagem ausente ou caminho inválido: {row["path"]}')
    if row['breed'] not in BREEDS or row['coat_pattern'] not in PATTERNS or row['coat_length'] not in COAT_LENGTHS:
        raise ValueError(f'Linha {line}: raça, padrão ou comprimento fora da taxonomia.')
    if set(filter(None, row['features'].split('|'))) - set(FEATURES):
        raise ValueError(f'Linha {line}: característica inválida.')
    if set(filter(None, row['colors'].split('|'))) - set(COLORS):
        raise ValueError(f'Linha {line}: cor inválida.')
    if row['split'] not in ('train', 'val'):
        raise ValueError(f'Linha {line}: split deve ser train ou val.')
counts = Counter(row['split'] for row in rows)
if not counts['train'] or not counts['val']:
    raise ValueError('O dataset precisa conter imagens train e val.')
print('Rótulos válidos:', dict(counts))


## Treine e baixe o modelo

O EfficientNet-B0 usa pesos pré-treinados do torchvision. O primeiro treino pode baixar esses pesos. Ajuste épocas e lote conforme a memória disponível. O arquivo gerado terá as cabeças exigidas pelo backend.


In [ ]:
EPOCHS = 10
BATCH_SIZE = 16
MODEL_PATH = Path('/content/catcare.pt')
subprocess.run([sys.executable, '-m', 'app.ml.train', '--csv', str(CSV_PATH), '--output', str(MODEL_PATH), '--epochs', str(EPOCHS), '--batch-size', str(BATCH_SIZE)], cwd=REPO / 'backend', check=True)
print('Modelo salvo em:', MODEL_PATH)


In [ ]:
from google.colab import files
files.download(str(MODEL_PATH))


Depois de baixar `catcare.pt`, coloque-o em um caminho acessível ao backend, configure `ML_MODE=real` e `ML_MODEL_PATH` no `.env` local. Não suba o checkpoint, o dataset ou o `.env` ao GitHub.
